# Step 8 — Bird's-Eye-View Fusion Video

| | |
|---|---|
| **Input** | `output/step_0` (samples index, ego pose), `output/step_1/lidar` + `output/step_2/lidar` (LiDAR clusters), `output/step_2/radar` (post-declutter radar points), `output/step_2/yolo_mono` (monocular camera detections), `output/step_1/camera_meta.json` + raw `CAM_FRONT` JPEGs (`DATA_ROOT`), `output/step_4/fused_tracks_all.csv`, `output/step_5/ttc_fused.csv` |
| **Outputs** | `output/step_8/videos/<scene_name>.mp4` — one video per nuScenes scene (10 total) |
| | `output/step_8/bev_video_summary.csv` — per-scene frame count / fps / filename |
| **Used by** | Nothing downstream — this is a terminal visualization stage |

## What this notebook does

Renders one combined frame per sample: a **Bird's-Eye View** (BEV) panel on the
left showing every sensor's detections for that frame — LiDAR cluster centroids,
post-declutter radar points, and true-monocular (`camera_mono`) camera
detections — all transformed into the ego vehicle's own frame (ego at the
origin, heading pointing "up"), plus the **fused** tracks active that frame
with their fused TTC value labeled and color-coded by urgency (red < 3s,
orange < 6s, yellow < 10s, green otherwise, gray = no finite TTC yet). The
right panel shows the real `CAM_FRONT` image for that sample — the actual
input video feed, for a direct sanity check against the BEV reconstruction.

Frames are then encoded into one MP4 per scene (nuScenes v1.0-mini's 404
samples are 10 separate continuous driving clips, not one single recording —
see `CLAUDE.md`'s "Video scope" discussion), at a per-scene fps derived from
that scene's own real sample-to-sample elapsed time (`timestamp_us`), so each
video reproduces the original recording's actual temporal cadence rather than
an assumed constant.

This stage is purely for visual inspection/presentation — it reads only
already-committed pipeline outputs and writes nothing any other stage
consumes.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Imports and output directories
# ─────────────────────────────────────────────────────────────────
import json
import shutil
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from config import DATA_ROOT, STEP0_DIR, STEP1_DIR, STEP2_DIR, STEP4_DIR, STEP5_DIR, STEP8_DIR
from src.geometry import transform_matrix, global_points_to_camera
import matplotlib.patheffects as pe

FRAMES_DIR = STEP8_DIR / "frames"
VIDEOS_DIR = STEP8_DIR / "videos"

# FIXED (same idempotency convention as Steps 3.1/3.2/3.3/4): clear this
# stage's own output dir before regenerating, so a re-run never mixes an old
# video/frame set with a new one.
if STEP8_DIR.exists():
    shutil.rmtree(STEP8_DIR)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\u2705 Step 8 output dirs ready: {STEP8_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ Step 8 output dirs ready: F:\Sensor fusion Research\output\step_8


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Named constants
# ─────────────────────────────────────────────────────────────────

BEV_RANGE_M = 60.0          # ego-centered BEV plot extent, metres each side
LIDAR_POINT_SIZE = 4
RADAR_MARKER_SIZE = 60
CAMERA_MARKER_SIZE = 50
FUSED_MARKER_SIZE = 140

# TTC urgency color bands (seconds) -- purely a visual color-coding choice
# for this BEV, not a scoring threshold (Step 6's Good_pct uses its own).
TTC_RED_S = 3.0
TTC_ORANGE_S = 6.0
TTC_YELLOW_S = 10.0

CAM_PANEL_CHANNEL = "CAM_FRONT"   # the "video as it was in input" reference panel
CAMERA_CHANNELS = ["CAM_FRONT", "CAM_FRONT_LEFT", "CAM_FRONT_RIGHT",
                    "CAM_BACK", "CAM_BACK_LEFT", "CAM_BACK_RIGHT"]
RADAR_CHANNELS = ["RADAR_FRONT", "RADAR_FRONT_LEFT", "RADAR_FRONT_RIGHT",
                   "RADAR_BACK_LEFT", "RADAR_BACK_RIGHT"]

FPS_MIN, FPS_MAX = 1.0, 10.0   # clamp a scene's measured fps to a sane range

# Camera-panel overlay: capped to the N nearest (by ego distance) visible
# fused tracks, and drawn as a 2D bounding box rather than a point marker.
NEAREST_N_CAMERA = 6              # cap -- the camera panel gets cluttered with every track labeled
BOX_HALF_EXTENT_M = 1.2           # generic object footprint half-width/length, metres (not class-aware)
BOX_HEIGHT_M = 1.8                # generic object height, metres (not class-aware)

print(f"\u2705 BEV_RANGE_M={BEV_RANGE_M}m, TTC color bands: red<{TTC_RED_S}s, orange<{TTC_ORANGE_S}s, yellow<{TTC_YELLOW_S}s")

✅ BEV_RANGE_M=60.0m, TTC color bands: red<3.0s, orange<6.0s, yellow<10.0s


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Per-sample data loaders
#
# Every loader below returns points already in the GLOBAL frame (metres),
# using the same point_to_global-style transform chain already established
# in src/geometry.py and used throughout Steps 1-4 -- nothing new here,
# just reused for visualization instead of tracking.
# ─────────────────────────────────────────────────────────────────

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

with open(STEP1_DIR / "camera_meta.json") as f:
    camera_meta = json.load(f)

fused_tracks_df = pd.read_csv(STEP4_DIR / "fused_tracks_all.csv")
ttc_fused_df = pd.read_csv(STEP5_DIR / "ttc_fused.csv")[["fused_id", "sample_id", "ttc", "distance"]]
# Left join: a fused point with no TTC yet (e.g. a track's very first frame,
# before enough history exists to estimate velocity) keeps ttc=NaN rather
# than being dropped -- rendered as "no TTC yet" (gray), not silently hidden.
fused_with_ttc = fused_tracks_df.merge(ttc_fused_df, on=["fused_id", "sample_id"], how="left")


def _apply_transform(M, points_xyz):
    """points_xyz: (N,3) -> (N,3), via a precomputed 4x4 homogeneous matrix."""
    pts = np.asarray(points_xyz, dtype=float)
    if len(pts) == 0:
        return pts
    pts_h = np.hstack([pts, np.ones((len(pts), 1))]).T
    return (M @ pts_h)[:3].T


def load_ego_pose(sample_id):
    with open(STEP0_DIR / sample_id / "ego_pose.json") as f:
        return json.load(f)


def load_lidar_points_global(sample_id):
    """Non-ground LiDAR cluster centroids, lifted to the global frame."""
    clusters_path = STEP2_DIR / "lidar" / sample_id / "lidar_clusters.json"
    meta_path = STEP1_DIR / "lidar" / sample_id / "lidar_meta.json"
    if not clusters_path.exists() or not meta_path.exists():
        return np.empty((0, 3))
    with open(clusters_path) as f:
        clusters = json.load(f)
    if not clusters:
        return np.empty((0, 3))
    with open(meta_path) as f:
        lidar_meta = json.load(f)
    ego_pose = lidar_meta["ego_pose"]
    calib = {"translation": lidar_meta["sensor_to_ego_translation"],
              "rotation": lidar_meta["sensor_to_ego_rotation"]}
    T = transform_matrix(ego_pose["translation"], ego_pose["rotation"]) @ \
        transform_matrix(calib["translation"], calib["rotation"])
    centroids = np.array([v["centroid"] for v in clusters.values()])
    return _apply_transform(T, centroids)


def load_radar_points_global(sample_id):
    """All kept (post-declutter) radar points across all 5 channels, global frame."""
    out = []
    radar_dir = STEP2_DIR / "radar" / sample_id
    if not radar_dir.exists():
        return np.empty((0, 3))
    for chan in RADAR_CHANNELS:
        p = radar_dir / f"{chan}.json"
        if not p.exists():
            continue
        with open(p) as f:
            d = json.load(f)
        pts = d.get("points", [])
        if not pts:
            continue
        calib = d["calibration"]
        T = transform_matrix(calib["ego_pose"]["translation"], calib["ego_pose"]["rotation"]) @ \
            transform_matrix(calib["sensor_to_ego_translation"], calib["sensor_to_ego_rotation"])
        local = np.array([[pt["x"], pt["y"], pt["z"]] for pt in pts])
        out.append(_apply_transform(T, local))
    if not out:
        return np.empty((0, 3))
    return np.vstack(out)


def load_camera_mono_points_global(sample_id):
    """All 6 cameras' true-monocular (camera_mono) detections, already global."""
    out = []
    cam_dir = STEP2_DIR / "yolo_mono" / sample_id
    if not cam_dir.exists():
        return np.empty((0, 3))
    for chan in CAMERA_CHANNELS:
        p = cam_dir / f"{chan}.json"
        if not p.exists():
            continue
        with open(p) as f:
            dets = json.load(f)
        for det in dets:
            if det.get("has_3d_position"):
                out.append([det["global_x"], det["global_y"], det["global_z"]])
    if not out:
        return np.empty((0, 3))
    return np.array(out)


def load_fused_tracks_for_sample(sample_id):
    """Fused (x, y, z, ttc, distance) for every fused track active in this sample."""
    rows = fused_with_ttc[fused_with_ttc["sample_id"] == sample_id]
    return rows[["fused_id", "x", "y", "z", "ttc", "distance"]].to_dict("records")


def load_cam_front_image(sample_id):
    meta = camera_meta.get(sample_id, {}).get(CAM_PANEL_CHANNEL)
    if meta is None:
        return None
    path = DATA_ROOT / meta["filename"]
    if not path.exists():
        return None
    return Image.open(path)


print(f"\u2705 Loaded samples_index ({len(samples_index)}), camera_meta, fused_tracks+ttc ({len(fused_with_ttc)} rows)")

✅ Loaded samples_index (404), camera_meta, fused_tracks+ttc (40297 rows)


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Rendering: BEV panel, camera panel, combined frame
# ─────────────────────────────────────────────────────────────────

def ttc_to_color(ttc):
    if ttc is None or (isinstance(ttc, float) and (np.isnan(ttc) or np.isinf(ttc))):
        return "#888888"
    if ttc < TTC_RED_S:
        return "#d62728"
    if ttc < TTC_ORANGE_S:
        return "#ff7f0e"
    if ttc < TTC_YELLOW_S:
        return "#bcbd22"
    return "#2ca02c"


def to_ego_frame(points_global, ego_pose):
    """GLOBAL -> ego-local frame (ego at origin, x=forward, y=left), the
    same transform chain as src/geometry.py's point_to_global(), inverted."""
    if len(points_global) == 0:
        return np.empty((0, 3))
    T_inv = transform_matrix(ego_pose["translation"], ego_pose["rotation"], inverse=True)
    return _apply_transform(T_inv, points_global)


def render_bev(ax, sample_id, ego_pose):
    r = BEV_RANGE_M
    ax.set_xlim(-r, r)
    ax.set_ylim(-r, r)
    ax.set_aspect("equal")
    ax.set_facecolor("#0d1117")
    ax.grid(True, color="#30363d", linewidth=0.5)

    lidar_e = to_ego_frame(load_lidar_points_global(sample_id), ego_pose)
    if len(lidar_e):
        ax.scatter(-lidar_e[:, 1], lidar_e[:, 0], s=LIDAR_POINT_SIZE, c="#8b949e",
                   alpha=0.6, label="LiDAR", zorder=2)

    radar_e = to_ego_frame(load_radar_points_global(sample_id), ego_pose)
    if len(radar_e):
        ax.scatter(-radar_e[:, 1], radar_e[:, 0], s=RADAR_MARKER_SIZE, c="#ff7f0e",
                   marker="^", edgecolors="none", alpha=0.85, label="Radar", zorder=3)

    cam_e = to_ego_frame(load_camera_mono_points_global(sample_id), ego_pose)
    if len(cam_e):
        ax.scatter(-cam_e[:, 1], cam_e[:, 0], s=CAMERA_MARKER_SIZE, c="#58a6ff",
                   marker="s", edgecolors="none", alpha=0.85, label="Camera (mono)", zorder=3)

    for row in load_fused_tracks_for_sample(sample_id):
        pos_e = to_ego_frame(np.array([[row["x"], row["y"], 0.0]]), ego_pose)[0]
        px, py = -pos_e[1], pos_e[0]
        if abs(px) > r or abs(py) > r:
            continue
        ttc = row["ttc"]
        color = ttc_to_color(ttc)
        ax.scatter([px], [py], s=FUSED_MARKER_SIZE, facecolors="none", edgecolors=color,
                   linewidths=2.2, zorder=4)
        label = f"{ttc:.1f}s" if ttc is not None and not (np.isnan(ttc) or np.isinf(ttc)) else "\u2014"
        ax.annotate(label, (px, py), textcoords="offset points", xytext=(6, 6),
                    fontsize=8, color=color, fontweight="bold")

    ego_tri = plt.Polygon([(0, 2.2), (-1.3, -1.3), (1.3, -1.3)], closed=True,
                           color="#f0f6fc", zorder=5)
    ax.add_patch(ego_tri)

    ax.set_xlabel("lateral (m)", color="#c9d1d9")
    ax.set_ylabel("forward (m)", color="#c9d1d9")
    ax.tick_params(colors="#c9d1d9")
    for spine in ax.spines.values():
        spine.set_color("#30363d")
    ax.set_title("Bird's-Eye View - all sensors + fused TTC", color="#f0f6fc", fontsize=11)
    ax.legend(loc="upper right", fontsize=7, facecolor="#161b22", labelcolor="#c9d1d9", framealpha=0.85)


def render_camera_panel(ax, sample_id):
    img = load_cam_front_image(sample_id)
    ax.set_facecolor("#0d1117")
    if img is not None:
        ax.imshow(img)
        _draw_fused_ttc_on_camera(ax, sample_id, img.size)
    else:
        ax.text(0.5, 0.5, f"{CAM_PANEL_CHANNEL} image not found", ha="center", va="center",
                 color="#c9d1d9", transform=ax.transAxes)
    ax.set_title(f"{CAM_PANEL_CHANNEL} - input video + fused TTC", color="#f0f6fc", fontsize=11)
    ax.axis("off")


def _project_box_to_camera(center_xyz, cam_meta, img_w, img_h):
    """A generic axis-aligned 3D box (BOX_HALF_EXTENT_M x BOX_HALF_EXTENT_M x
    BOX_HEIGHT_M, centered on the track's own position) projected to a 2D
    pixel bounding box -- not class-aware (fusion doesn't carry a class
    through), so every object gets the same generic footprint/height rather
    than a per-class size. Returns (umin, vmin, umax, vmax) using whichever
    of the 8 corners actually project inside the frame, or None if none do."""
    cx, cy, cz = center_xyz
    h = BOX_HALF_EXTENT_M
    hz = BOX_HEIGHT_M / 2.0
    corners = np.array([[cx + sx * h, cy + sy * h, cz + sz * hz]
                         for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)])
    u, v, depth, valid = global_points_to_camera(
        corners, cam_meta["ego_pose"], cam_meta["calibrated_sensor"],
        cam_meta["calibrated_sensor"]["camera_intrinsic"],
        image_width=img_w, image_height=img_h)
    if not valid.any():
        return None
    return float(u[valid].min()), float(v[valid].min()), float(u[valid].max()), float(v[valid].max())


def _draw_fused_ttc_on_camera(ax, sample_id, img_size):
    """Projects the nearest NEAREST_N_CAMERA active fused tracks' positions
    into this sample's CAM_FRONT image plane (reusing src/geometry.py's
    global_points_to_camera, the same projection chain already used by Step
    2.3.1/3.1/3.2 -- no new math), draws a bounding box for each, and labels
    it with the fused TTC, same color coding as the BEV. Capped to the
    nearest few objects (by the same ego-distance Step 5 already computes
    for TTC) rather than every track, since labeling all of them makes
    distant, tightly-clustered objects near the image's vanishing point
    unreadable."""
    cam_meta = camera_meta.get(sample_id, {}).get(CAM_PANEL_CHANNEL)
    if cam_meta is None:
        return
    tracks = load_fused_tracks_for_sample(sample_id)
    if not tracks:
        return
    img_w, img_h = img_size

    centers = np.array([[t["x"], t["y"], t["z"]] for t in tracks])
    u, v, depth, valid = global_points_to_camera(
        centers, cam_meta["ego_pose"], cam_meta["calibrated_sensor"],
        cam_meta["calibrated_sensor"]["camera_intrinsic"],
        image_width=img_w, image_height=img_h)

    visible = [t for t, ok in zip(tracks, valid) if ok]
    visible.sort(key=lambda t: t["distance"] if t["distance"] is not None and not np.isnan(t["distance"]) else np.inf)
    nearest = visible[:NEAREST_N_CAMERA]

    outline = [pe.withStroke(linewidth=2.5, foreground="black")]
    for t in nearest:
        box = _project_box_to_camera((t["x"], t["y"], t["z"]), cam_meta, img_w, img_h)
        if box is None:
            continue
        umin, vmin, umax, vmax = box
        ttc = t["ttc"]
        color = ttc_to_color(ttc)
        label = f"{ttc:.1f}s" if ttc is not None and not (np.isnan(ttc) or np.isinf(ttc)) else "\u2014"
        rect = plt.Rectangle((umin, vmin), umax - umin, vmax - vmin, fill=False,
                              edgecolor=color, linewidth=2.5, zorder=5)
        ax.add_patch(rect)
        txt = ax.annotate(label, (umin, vmin), textcoords="offset points", xytext=(2, 6),
                           fontsize=11, color=color, fontweight="bold", zorder=6)
        txt.set_path_effects(outline)


def render_frame(sample_id, out_path):
    ego_pose = load_ego_pose(sample_id)
    fig, (ax_bev, ax_cam) = plt.subplots(1, 2, figsize=(14, 6.5), dpi=110, facecolor="#0d1117")
    render_bev(ax_bev, sample_id, ego_pose)
    render_camera_panel(ax_cam, sample_id)
    fig.suptitle(sample_id, color="#f0f6fc", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_path, facecolor=fig.get_facecolor())
    plt.close(fig)


print("\u2705 Rendering functions ready")

✅ Rendering functions ready


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Main loop: render every sample, encode one MP4 per scene
# ─────────────────────────────────────────────────────────────────

scenes = OrderedDict()
for sid in sorted(samples_index.keys()):
    scenes.setdefault(samples_index[sid]["scene_name"], []).append(sid)

summary_rows = []

for scene_name, sample_ids in scenes.items():
    scene_frames_dir = FRAMES_DIR / scene_name
    scene_frames_dir.mkdir(parents=True, exist_ok=True)

    frame_paths = []
    for sid in tqdm(sample_ids, desc=f"Rendering {scene_name}"):
        out_path = scene_frames_dir / f"{sid}.png"
        render_frame(sid, out_path)
        frame_paths.append(out_path)

    # fps derived from this scene's own real elapsed time between samples
    # (timestamp_us) -- reproduces the original recording's actual temporal
    # cadence rather than assuming a fixed 2Hz.
    timestamps = [samples_index[sid]["timestamp_us"] for sid in sample_ids]
    dts = np.diff(timestamps) / 1e6
    dts = dts[dts > 0]
    fps = float(np.clip(1.0 / np.median(dts), FPS_MIN, FPS_MAX)) if len(dts) else 2.0

    first_frame = cv2.imread(str(frame_paths[0]))
    h, w = first_frame.shape[:2]
    video_path = VIDEOS_DIR / f"{scene_name}.mp4"
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for fp in frame_paths:
        writer.write(cv2.imread(str(fp)))
    writer.release()

    shutil.rmtree(scene_frames_dir)   # frames were only scratch for encoding

    summary_rows.append({"scene_name": scene_name, "n_samples": len(sample_ids),
                          "fps": round(fps, 2), "video": video_path.name})
    print(f"\u2705 {scene_name}: {len(sample_ids)} frames @ {fps:.2f} fps -> {video_path.name}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(STEP8_DIR / "bev_video_summary.csv", index=False)
print(f"\n\u2705 Step 8 complete: {len(summary_rows)} scene videos written to {VIDEOS_DIR}")
summary_df

Rendering scene-0061:   0%|          | 0/39 [00:00<?, ?it/s]

Rendering scene-0061:   3%|▎         | 1/39 [00:00<00:22,  1.66it/s]

Rendering scene-0061:   5%|▌         | 2/39 [00:01<00:22,  1.61it/s]

Rendering scene-0061:   8%|▊         | 3/39 [00:01<00:22,  1.58it/s]

Rendering scene-0061:  10%|█         | 4/39 [00:02<00:22,  1.55it/s]

...

Rendering scene-1100:  98%|█████████▊| 39/40 [00:19<00:00,  2.06it/s]

Rendering scene-1100: 100%|██████████| 40/40 [00:19<00:00,  2.10it/s]

Rendering scene-1100: 100%|██████████| 40/40 [00:19<00:00,  2.05it/s]

✅ scene-1100: 40 frames @ 2.00 fps -> scene-1100.mp4

✅ Step 8 complete: 10 scene videos written to F:\Sensor fusion Research\output\step_8\videos


,scene_name,n_samples,fps,video
0,scene-0061,39,2.0,scene-0061.mp4
1,scene-0103,40,2.0,scene-0103.mp4
2,scene-0553,41,2.0,scene-0553.mp4
3,scene-0655,41,2.0,scene-0655.mp4
4,scene-0757,41,2.0,scene-0757.mp4
5,scene-0796,40,2.0,scene-0796.mp4
6,scene-0916,41,2.0,scene-0916.mp4
7,scene-1077,41,2.0,scene-1077.mp4
8,scene-1094,40,2.0,scene-1094.mp4
9,scene-1100,40,2.0,scene-1100.mp4


## Part 2 — the "mission control" dashboard video (6 cameras + radar-style BEV)

A second, more detailed per-scene video, added on request to match a provided
reference layout: a 2x3 grid of all 6 camera views (not just `CAM_FRONT`),
each with small TTC/distance-labeled markers on the nearest fused tracks
visible in it, next to a radar-style polar BEV panel — range rings, 6
compass-style sector labels (`FRONT`/`FRONT_RIGHT`/`BACK_RIGHT`/`BACK`/
`BACK_LEFT`/`FRONT_LEFT`, matching the radar channel names), a blue
to-scale ego rectangle, yellow velocity arrows, color-coded TTC dots, and
the real nuScenes ground-truth boxes (green) for direct visual comparison
against the fused prediction. A header line reports the scene, sample,
ego speed, and a "critical tracks" count (fused tracks with TTC < 4s).

This is a separate, additional output (`output/step_8/videos_detailed/`)
— Part 1's simpler 2-panel video above is unchanged.

In [6]:
# ─────────────────────────────────────────────────────────────────
# CELL 7 — Ground-truth boxes, ego speed, and a new TTC color scale
# matching the reference layout's own legend (red<2s, orange 2-4s,
# yellow 4-10s, gray = not closing) -- distinct from Part 1's bins.
# ─────────────────────────────────────────────────────────────────
from nuscenes.nuscenes import NuScenes
from pyquaternion import Quaternion

from config import NUSCENES_VERSION

nusc = NuScenes(version=NUSCENES_VERSION, dataroot=str(DATA_ROOT), verbose=False)

DETAILED_VIDEOS_DIR = STEP8_DIR / "videos_detailed"
DETAILED_FRAMES_DIR = STEP8_DIR / "frames_detailed"
DETAILED_VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
DETAILED_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

TTC2_RED_S = 2.0
TTC2_ORANGE_S = 4.0
TTC2_YELLOW_S = 10.0
CRITICAL_TTC_S = 4.0   # fused tracks with ttc < this count toward the header's "critical tracks"

NEAREST_N_PER_CAMERA = 6
ARROW_LOOKAHEAD_S = 1.5         # velocity arrow length = this many seconds of travel
EGO_BOX_LENGTH_M, EGO_BOX_WIDTH_M = 4.5, 1.9   # generic passenger-car footprint for the ego rectangle

SECTOR_LABELS = [("FRONT", 0), ("FRONT_RIGHT", 60), ("BACK_RIGHT", 120),
                  ("BACK", 180), ("BACK_LEFT", 240), ("FRONT_LEFT", 300)]


def ttc_to_color_v2(ttc):
    if ttc is None or (isinstance(ttc, float) and (np.isnan(ttc) or np.isinf(ttc))):
        return "#9aa4af"
    if ttc < TTC2_RED_S:
        return "#e5484d"
    if ttc < TTC2_ORANGE_S:
        return "#f5a623"
    if ttc < TTC2_YELLOW_S:
        return "#e8e23d"
    return "#9aa4af"


def load_gt_boxes_ego(sample_id, ego_pose):
    """Real nuScenes sample_annotation boxes for this sample, as (x_ego, y_ego,
    yaw_ego, length, width) tuples already in the ego-local, heading-up frame
    -- same global->ego transform as everything else in this notebook."""
    sample = nusc.get("sample", samples_index[sample_id]["sample_token"])
    T_inv = transform_matrix(ego_pose["translation"], ego_pose["rotation"], inverse=True)
    boxes = []
    for ann_token in sample["anns"]:
        ann = nusc.get("sample_annotation", ann_token)
        # Match the pipeline's own tracked classes (YOLO: person/bicycle/car/
        # motorcycle/bus/truck) -- otherwise movable_object.barrier/trafficcone
        # (very common in this construction-zone scene: 52 + 37 vs. ~59 real
        # vehicles/pedestrians) swamp the plot with clutter that was never a
        # fusion/TTC target to begin with.
        cat = ann["category_name"]
        if not (cat.startswith("vehicle.") or cat.startswith("human.pedestrian.")):
            continue
        w, l, h = ann["size"]
        yaw_global = Quaternion(ann["rotation"]).yaw_pitch_roll[0]
        center_global = np.array(ann["translation"])
        center_ego = _apply_transform(T_inv, center_global.reshape(1, 3))[0]
        # Rotate the box's heading by the same amount the ego frame itself
        # rotates away from global -- i.e. subtract ego's own global yaw.
        ego_yaw_global = Quaternion(ego_pose["rotation"]).yaw_pitch_roll[0]
        yaw_ego = yaw_global - ego_yaw_global
        boxes.append((center_ego[0], center_ego[1], yaw_ego, l, w))
    return boxes


def ego_speed_kmh(sample_id, scene_sample_ids):
    """Ego speed from the real translation delta to the previous sample in
    the SAME scene (falls back to the next sample at a scene's first frame)."""
    idx = scene_sample_ids.index(sample_id)
    if idx > 0:
        s0, s1 = scene_sample_ids[idx - 1], sample_id
    elif len(scene_sample_ids) > 1:
        s0, s1 = sample_id, scene_sample_ids[idx + 1]
    else:
        return 0.0
    p0 = np.array(load_ego_pose(s0)["translation"][:2])
    p1 = np.array(load_ego_pose(s1)["translation"][:2])
    dt = (samples_index[s1]["timestamp_us"] - samples_index[s0]["timestamp_us"]) / 1e6
    if dt <= 0:
        return 0.0
    return float(np.linalg.norm(p1 - p0) / dt * 3.6)


def rotate_vector_to_ego(vx, vy, ego_pose):
    T_inv = transform_matrix(ego_pose["translation"], ego_pose["rotation"], inverse=True)
    R_inv = T_inv[:3, :3]
    return (R_inv @ np.array([vx, vy, 0.0]))[:2]


print(f"\u2705 NuScenes devkit loaded, GT/ego-speed/TTC-v2 helpers ready")

✅ NuScenes devkit loaded, GT/ego-speed/TTC-v2 helpers ready


In [7]:
# ─────────────────────────────────────────────────────────────────
# CELL 8 — Polar/radar-style BEV panel
# ─────────────────────────────────────────────────────────────────

def render_radar_bev(ax, sample_id, ego_pose, scene_name, scene_sample_ids):
    r = BEV_RANGE_M
    ax.set_xlim(-r * 1.18, r * 1.18)
    ax.set_ylim(-r * 1.1, r * 1.18)
    ax.set_aspect("equal")
    ax.set_facecolor("#05080c")
    ax.axis("off")

    # Range rings + spokes at the 6 sector angles
    for ring_r in np.linspace(r / 4, r, 4):
        circle = plt.Circle((0, 0), ring_r, fill=False, color="#2a3540", linewidth=0.8, zorder=1)
        ax.add_patch(circle)
    for _, bearing in SECTOR_LABELS:
        theta = np.radians(bearing)
        ax.plot([0, r * np.sin(theta)], [0, r * np.cos(theta)], color="#1c242c", linewidth=0.8, zorder=1)
    for name, bearing in SECTOR_LABELS:
        theta = np.radians(bearing)
        lx, ly = r * 1.1 * np.sin(theta), r * 1.1 * np.cos(theta)
        ax.text(lx, ly, name, color="#4e5b68", fontsize=7, ha="center", va="center", zorder=2)

    # Ground-truth boxes (green, dashed)
    for (cx, cy, yaw, length, width) in load_gt_boxes_ego(sample_id, ego_pose):
        if abs(cx) > r * 1.3 or abs(cy) > r * 1.3:
            continue
        corners_local = np.array([[length / 2, width / 2], [length / 2, -width / 2],
                                    [-length / 2, -width / 2], [-length / 2, width / 2]])
        c, s = np.cos(yaw), np.sin(yaw)
        rot = np.array([[c, -s], [s, c]])
        corners_ego = (rot @ corners_local.T).T + np.array([cx, cy])
        plot_pts = np.column_stack([-corners_ego[:, 1], corners_ego[:, 0]])
        poly = plt.Polygon(plot_pts, closed=True, fill=False, edgecolor="#2ecc71",
                            linewidth=1.1, linestyle="--", zorder=2, alpha=0.85)
        ax.add_patch(poly)

    # Fused tracks: velocity arrow + TTC-colored dot + label
    n_critical = 0
    for row in load_fused_tracks_for_sample(sample_id):
        pos_e = to_ego_frame(np.array([[row["x"], row["y"], 0.0]]), ego_pose)[0]
        px, py = -pos_e[1], pos_e[0]
        if abs(px) > r or abs(py) > r:
            continue
        ttc = row["ttc"]
        if ttc is not None and not (np.isnan(ttc) or np.isinf(ttc)) and ttc < CRITICAL_TTC_S:
            n_critical += 1
        color = ttc_to_color_v2(ttc)

        track_row = fused_with_ttc[(fused_with_ttc["fused_id"] == row["fused_id"]) &
                                     (fused_with_ttc["sample_id"] == sample_id)]
        if len(track_row):
            vx, vy = float(track_row["vx"].iloc[0]), float(track_row["vy"].iloc[0])
            vxe, vye = rotate_vector_to_ego(vx, vy, ego_pose)
            dpx, dpy = -vye * ARROW_LOOKAHEAD_S, vxe * ARROW_LOOKAHEAD_S
            if abs(dpx) > 0.3 or abs(dpy) > 0.3:
                ax.annotate("", xy=(px + dpx, py + dpy), xytext=(px, py),
                            arrowprops=dict(arrowstyle="-|>", color="#e8c13d", lw=1.4, alpha=0.9), zorder=3)

        ax.scatter([px], [py], s=70, c=color, edgecolors="#05080c", linewidths=0.8, zorder=4)
        label = f"{ttc:.1f}s" if ttc is not None and not (np.isnan(ttc) or np.isinf(ttc)) else "\u2014"
        ax.annotate(label, (px, py), textcoords="offset points", xytext=(5, 5),
                    fontsize=7, color=color, fontweight="bold", zorder=5)

    # Ego, to scale, heading up
    hl, hw = EGO_BOX_LENGTH_M / 2, EGO_BOX_WIDTH_M / 2
    ego_rect = plt.Polygon([(-hw, -hl), (hw, -hl), (hw, hl), (-hw, hl)], closed=True,
                            facecolor="#4f8ff0", edgecolor="#bcd6ff", linewidth=1, zorder=6)
    ax.add_patch(ego_rect)

    # Legend (manual, bottom-left)
    legend_items = [("TTC < 2 s", "#e5484d"), ("2-4 s", "#f5a623"), ("4-10 s", "#e8e23d"),
                     ("not closing", "#9aa4af"), ("GT box", "#2ecc71")]
    ly0 = -r * 1.02
    for i, (txt, col) in enumerate(legend_items):
        ly = ly0 + i * (r * 0.09)
        ax.scatter([-r * 1.12], [ly], s=35, c=col, zorder=7)
        ax.text(-r * 1.05, ly, txt, color="#aab4bf", fontsize=6.5, va="center", zorder=7)

    # Header
    speed_kmh = ego_speed_kmh(sample_id, scene_sample_ids)
    header = f"{scene_name} | {sample_id} | ego {speed_kmh:.0f} km/h | critical tracks: {n_critical}"
    ax.text(0, r * 1.26, header, color="#e8c13d", fontsize=9.5, ha="center", va="bottom", zorder=8)


print("\u2705 Polar BEV renderer ready")

✅ Polar BEV renderer ready


In [8]:
# ─────────────────────────────────────────────────────────────────
# CELL 9 — 6-camera grid, each with its own nearest-N TTC/distance labels
# ─────────────────────────────────────────────────────────────────

CAMERA_GRID_ORDER = [["CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT"],
                      ["CAM_BACK_LEFT", "CAM_BACK", "CAM_BACK_RIGHT"]]


def render_camera_cell(ax, sample_id, channel, tracks):
    meta = camera_meta.get(sample_id, {}).get(channel)
    ax.set_facecolor("#05080c")
    ax.axis("off")
    ax.set_title(channel, color="#8b949e", fontsize=8)
    if meta is None:
        ax.text(0.5, 0.5, "no image", ha="center", va="center", color="#8b949e", transform=ax.transAxes)
        return
    path = DATA_ROOT / meta["filename"]
    if not path.exists():
        ax.text(0.5, 0.5, "image not found", ha="center", va="center", color="#8b949e", transform=ax.transAxes)
        return
    img = Image.open(path)
    ax.imshow(img)
    img_w, img_h = img.size

    if not tracks:
        return
    centers = np.array([[t["x"], t["y"], t["z"]] for t in tracks])
    u, v, depth, valid = global_points_to_camera(
        centers, meta["ego_pose"], meta["calibrated_sensor"],
        meta["calibrated_sensor"]["camera_intrinsic"], image_width=img_w, image_height=img_h)

    visible = [(t, u[i], v[i]) for i, t in enumerate(tracks) if valid[i]]
    visible.sort(key=lambda r: r[0]["distance"] if r[0]["distance"] is not None and not np.isnan(r[0]["distance"]) else np.inf)
    outline = [pe.withStroke(linewidth=1.8, foreground="black")]
    for t, uu, vv in visible[:NEAREST_N_PER_CAMERA]:
        ttc, dist = t["ttc"], t["distance"]
        color = ttc_to_color_v2(ttc)
        parts = []
        if ttc is not None and not (np.isnan(ttc) or np.isinf(ttc)):
            parts.append(f"{ttc:.1f}s")
        if dist is not None and not np.isnan(dist):
            parts.append(f"{dist:.0f}m")
        label = " ".join(parts) if parts else "\u2014"
        ax.scatter([uu], [vv], s=45, facecolors="none", edgecolors=color, linewidths=1.6, zorder=5)
        txt = ax.annotate(label, (uu, vv), textcoords="offset points", xytext=(5, 5),
                           fontsize=6.5, color=color, fontweight="bold", zorder=6)
        txt.set_path_effects(outline)


print("\u2705 6-camera grid renderer ready")

✅ 6-camera grid renderer ready


In [9]:
# ─────────────────────────────────────────────────────────────────
# CELL 10 — Combine into one frame, render every sample, encode per-scene video
# ─────────────────────────────────────────────────────────────────

def render_detailed_frame(sample_id, scene_name, scene_sample_ids, out_path):
    ego_pose = load_ego_pose(sample_id)
    tracks = load_fused_tracks_for_sample(sample_id)

    fig = plt.figure(figsize=(20, 9), dpi=100, facecolor="#05080c")
    gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 1.35], wspace=0.04, hspace=0.12,
                            left=0.01, right=0.99, top=0.96, bottom=0.02)

    for r_i, row in enumerate(CAMERA_GRID_ORDER):
        for c_i, channel in enumerate(row):
            ax = fig.add_subplot(gs[r_i, c_i])
            render_camera_cell(ax, sample_id, channel, tracks)

    bev_ax = fig.add_subplot(gs[:, 3])
    render_radar_bev(bev_ax, sample_id, ego_pose, scene_name, scene_sample_ids)

    fig.savefig(out_path, facecolor=fig.get_facecolor())
    plt.close(fig)


detailed_summary_rows = []

for scene_name, sample_ids in scenes.items():
    scene_frames_dir = DETAILED_FRAMES_DIR / scene_name
    scene_frames_dir.mkdir(parents=True, exist_ok=True)

    frame_paths = []
    for sid in tqdm(sample_ids, desc=f"Rendering (detailed) {scene_name}"):
        out_path = scene_frames_dir / f"{sid}.png"
        render_detailed_frame(sid, scene_name, sample_ids, out_path)
        frame_paths.append(out_path)

    timestamps = [samples_index[sid]["timestamp_us"] for sid in sample_ids]
    dts = np.diff(timestamps) / 1e6
    dts = dts[dts > 0]
    fps = float(np.clip(1.0 / np.median(dts), FPS_MIN, FPS_MAX)) if len(dts) else 2.0

    first_frame = cv2.imread(str(frame_paths[0]))
    h, w = first_frame.shape[:2]
    video_path = DETAILED_VIDEOS_DIR / f"{scene_name}.mp4"
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for fp in frame_paths:
        writer.write(cv2.imread(str(fp)))
    writer.release()

    shutil.rmtree(scene_frames_dir)

    detailed_summary_rows.append({"scene_name": scene_name, "n_samples": len(sample_ids),
                                    "fps": round(fps, 2), "video": video_path.name})
    print(f"\u2705 {scene_name}: {len(sample_ids)} frames @ {fps:.2f} fps -> {video_path.name}")

shutil.rmtree(DETAILED_FRAMES_DIR)

detailed_summary_df = pd.DataFrame(detailed_summary_rows)
detailed_summary_df.to_csv(STEP8_DIR / "detailed_video_summary.csv", index=False)
print(f"\n\u2705 Detailed dashboard video complete: {len(detailed_summary_rows)} scene videos written to {DETAILED_VIDEOS_DIR}")
detailed_summary_df

Rendering (detailed) scene-0061:   0%|          | 0/39 [00:00<?, ?it/s]

Rendering (detailed) scene-0061:   3%|▎         | 1/39 [00:01<00:51,  1.36s/it]

Rendering (detailed) scene-0061:   5%|▌         | 2/39 [00:02<00:53,  1.45s/it]

Rendering (detailed) scene-0061:   8%|▊         | 3/39 [00:04<00:56,  1.57s/it]

Rendering (detailed) scene-0061:  10%|█         | 4/39 [00:06<00:56,  1.62s/it]

...

Rendering (detailed) scene-1100:  98%|█████████▊| 39/40 [00:56<00:01,  1.38s/it]

Rendering (detailed) scene-1100: 100%|██████████| 40/40 [00:58<00:00,  1.43s/it]

Rendering (detailed) scene-1100: 100%|██████████| 40/40 [00:58<00:00,  1.46s/it]

✅ scene-1100: 40 frames @ 2.00 fps -> scene-1100.mp4

✅ Detailed dashboard video complete: 10 scene videos written to F:\Sensor fusion Research\output\step_8\videos_detailed


,scene_name,n_samples,fps,video
0,scene-0061,39,2.0,scene-0061.mp4
1,scene-0103,40,2.0,scene-0103.mp4
2,scene-0553,41,2.0,scene-0553.mp4
3,scene-0655,41,2.0,scene-0655.mp4
4,scene-0757,41,2.0,scene-0757.mp4
5,scene-0796,40,2.0,scene-0796.mp4
6,scene-0916,41,2.0,scene-0916.mp4
7,scene-1077,41,2.0,scene-1077.mp4
8,scene-1094,40,2.0,scene-1094.mp4
9,scene-1100,40,2.0,scene-1100.mp4
